# Exhaustive Per-Shot Correlation Analysis: Logical Error Distribution vs Fixed-Class LLRs

This notebook performs **exhaustive correlation analysis** by exploring ALL logical classes (2^k)
for each shot. This provides complete data for analyzing whether the logical error distribution
predicts which error class will have the lowest fixed-class LLR.

Key differences from sampled analysis:
- **Complete coverage**: Explores all 2^k logical classes, not just sampled representatives
- **Parallelization**: Uses `num_procs_for_gap` for efficient parallel decoding
- **Fewer shots**: Due to computational cost, typically 10 shots vs 1000+ in sampled mode

Key metrics:
1. **Per-shot Spearman correlation**: Does rank predict LLR ordering within each shot?
2. **Optimal selection rate**: How often does the highest-probability error have the lowest LLR?
3. **Complete top-k success curve**: P(winner in top-k) for all k values
4. **Winning rank distribution**: Which ranks win most often?

## 1. Data Collection Configuration

Configure the simulation parameters below, then run the cell to generate the script command.

In [68]:
# ==============================================================================
# CONFIGURATION - Set your simulation parameters here
# ==============================================================================

# Code parameters
N_QUBITS = 144          # BB code variant: 72, 90, 108, 144, 288, 360, 756
P = 0.003               # Physical error rate

# Simulation parameters (exhaustive mode)
N_SHOTS = 10           # Number of shots to simulate (keep small, exhaustive is expensive)
NUM_PROCS_FOR_GAP = 126 # Parallel processes for gap computation
PARALLEL_VERBOSE = 3   # Joblib verbosity (0=silent, higher=more progress info)
SEED = 42               # Random seed

# Output path (auto-generated based on parameters)
OUTPUT_DIR = "simulations/data/correlation_analysis"

In [69]:
# ==============================================================================
# Generate script command based on configuration
# ==============================================================================

from pathlib import Path

# Generate output filename
OUTPUT_FILENAME = f"bb_n{N_QUBITS}_p{P}_exhaustive_shots{N_SHOTS}.csv"
OUTPUT_PATH = Path(OUTPUT_DIR) / OUTPUT_FILENAME

# Build command
SCRIPT_PATH = "simulations/analysis/scripts/run_distribution_correlation_analysis.py"

cmd_parts = [
    f"python {SCRIPT_PATH}",
    f"    --n-qubits {N_QUBITS}",
    f"    --p {P}",
    f"    --n-shots {N_SHOTS}",
    f"    --exhaustive",
    f"    --num-procs-for-gap {NUM_PROCS_FOR_GAP}",
    f"    --parallel-verbose {PARALLEL_VERBOSE}",
    f"    --seed {SEED}",
    f"    --output {OUTPUT_PATH}",
]

command = " \\\n".join(cmd_parts)

# Display
print("=" * 70)
print("GENERATED COMMAND (Exhaustive Mode)")
print("=" * 70)
print()
print(command)
print()
print("=" * 70)
print(f"Output file: {OUTPUT_PATH}")
print("=" * 70)

# Store for later use
RESULTS_PATH = Path("../../..") / OUTPUT_PATH

GENERATED COMMAND (Exhaustive Mode)

python simulations/analysis/scripts/run_distribution_correlation_analysis.py \
    --n-qubits 144 \
    --p 0.003 \
    --n-shots 10 \
    --exhaustive \
    --num-procs-for-gap 126 \
    --parallel-verbose 3 \
    --seed 42 \
    --output simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive_shots10.csv

Output file: simulations/data/correlation_analysis/bb_n144_p0.003_exhaustive_shots10.csv


In [70]:
# Autoreload extension for Jupyter notebooks
%load_ext autoreload
%autoreload 2

# Standard library imports
import json
from pathlib import Path

# Third-party library imports
import numpy as np
import pandas as pd
from scipy import stats

# Plotly for interactive visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pandas display settings
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.6f}'.format)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load Results

In [71]:
# ==============================================================================
# Load results (uses RESULTS_PATH from configuration above)
# ==============================================================================

if not RESULTS_PATH.exists():
    print(f"Results file not found: {RESULTS_PATH.resolve()}")
    print("\nPlease run the data collection script first using the command above.")
    raise FileNotFoundError(f"Results not found at {RESULTS_PATH}")

# Load DataFrame
df_results = pd.read_csv(RESULTS_PATH)

# Load metadata
metadata_path = RESULTS_PATH.with_suffix(".json")
if metadata_path.exists():
    with open(metadata_path) as f:
        metadata = json.load(f)
else:
    metadata = {}

# Display summary
print(f"Loaded {len(df_results):,} records from {RESULTS_PATH.name}")
print(f"\nMetadata:")
for key, value in metadata.items():
    if isinstance(value, dict):
        print(f"  {key}: {value}")
    else:
        print(f"  {key}: {value}")

print(f"\nDataFrame preview:")
df_results.head(10)

Loaded 40,950 records from bb_n144_p0.003_exhaustive_shots10.csv

Metadata:
  n_qubits: 144
  distance: 12
  p: 0.003
  n_shots: 10
  exhaustive: True
  total_logical_classes: 4096
  num_observables: 12
  logical_error_rate: 0.009543945312500024
  distribution_shots: 4096000
  decoder_params: {'max_iter': 30, 'bp_method': 'minimum_sum', 'lsd_method': 'LSD_0', 'lsd_order': 0}
  seed: 42
  num_procs_for_gap: 126

DataFrame preview:


,shot_id,error_rank,error_index,error_prob,fixed_llr,best_llr,llr_delta
0,0,3813,1320,0.000026,563.828963,142.548704,421.280259
1,0,3018,3368,0.000051,1026.984451,142.548704,884.435747
2,0,2400,296,0.000077,483.310247,142.548704,340.761543
3,0,2020,2344,0.000102,845.394877,142.548704,702.846173
4,0,4081,1832,0.000000,458.328725,142.548704,315.780021
5,0,1377,3880,0.000153,390.157875,142.548704,247.609171
6,0,1410,808,0.000153,685.484916,142.548704,542.936211
7,0,2469,2856,0.000077,466.965091,142.548704,324.416387
8,0,179,1064,0.000972,515.527029,142.548704,372.978325
9,0,739,3112,0.000256,447.677324,142.548704,305.128620


In [ ]:
# ==============================================================================
# Data Summary
# ==============================================================================

n_shots = df_results['shot_id'].nunique()
n_classes_per_shot = len(df_results) // n_shots
total_logical_classes = metadata.get('total_logical_classes', n_classes_per_shot)

# Check for no-error case (error_index=0)
has_no_error = (df_results['error_index'] == 0).any()

# Find rank of no-error class
if has_no_error:
    no_error_rank = df_results[df_results['error_index'] == 0]['error_rank'].iloc[0]
    no_error_prob = df_results[df_results['error_index'] == 0]['error_prob'].iloc[0]
else:
    no_error_rank = None
    no_error_prob = None

print("=" * 60)
print("DATA SUMMARY")
print("=" * 60)
print(f"\nTotal shots: {n_shots}")
print(f"Classes per shot: {n_classes_per_shot}")
print(f"Total logical classes: {total_logical_classes}")
print(f"Total records: {len(df_results):,}")
print(f"\nNo-error class (index=0) included: {'Yes' if has_no_error else 'No'}")
if has_no_error:
    print(f"  - Rank: {no_error_rank} (0 = most common)")
    print(f"  - Probability: {no_error_prob:.4%}")
print(f"\nClass rank range: {df_results['error_rank'].min()} to {df_results['error_rank'].max()}")
print(f"LLR range: {df_results['fixed_llr'].min():.2f} to {df_results['fixed_llr'].max():.2f}")

In [ ]:
# ==============================================================================
# Decoding Success Statistics (if available)
# ==============================================================================

if "initial_success" in df_results.columns and "final_success" in df_results.columns:
    # Get per-shot success (same for all records in a shot)
    shot_success = df_results.groupby("shot_id").agg({
        "initial_success": "first",
        "final_success": "first",
    })
    
    initial_success_rate = shot_success["initial_success"].mean()
    final_success_rate = shot_success["final_success"].mean()
    
    # Count improvements and regressions
    improved = ((~shot_success["initial_success"]) & shot_success["final_success"]).sum()
    regressed = (shot_success["initial_success"] & (~shot_success["final_success"])).sum()
    unchanged = (shot_success["initial_success"] == shot_success["final_success"]).sum()
    
    print("=" * 60)
    print("DECODING SUCCESS COMPARISON")
    print("=" * 60)
    print(f"\nInitial BP+LSD success rate: {100*initial_success_rate:.2f}% ({int(shot_success['initial_success'].sum())}/{n_shots})")
    print(f"Final (comparative) success rate: {100*final_success_rate:.2f}% ({int(shot_success['final_success'].sum())}/{n_shots})")
    print(f"\nGap proxy exploration effect:")
    print(f"  Improved (wrong → correct): {improved} shots")
    print(f"  Regressed (correct → wrong): {regressed} shots")
    print(f"  Unchanged: {unchanged} shots")
    
    if initial_success_rate > 0:
        improvement_factor = final_success_rate / initial_success_rate
        print(f"\nImprovement factor: {improvement_factor:.2f}x")
else:
    print("Note: Decoding success columns not found in data.")

In [ ]:
# ==============================================================================
# Cluster Fraction LLR 2-Norm vs Logical Gap Analysis
# ==============================================================================

if "cluster_frac_llr_2norm" in df_results.columns:
    # Compute per-shot statistics including logical gap
    shot_stats_with_cluster = []
    
    for shot_id, group in df_results.groupby("shot_id"):
        # Get cluster fraction (same for all records in shot)
        cluster_frac = group["cluster_frac_llr_2norm"].iloc[0]
        best_llr = group["best_llr"].iloc[0]
        
        # Compute logical gap: difference between best and second-best LLR
        sorted_llrs = group["fixed_llr"].sort_values()
        min_llr = sorted_llrs.iloc[0]
        second_min_llr = sorted_llrs.iloc[1] if len(sorted_llrs) > 1 else min_llr
        logical_gap = second_min_llr - min_llr
        
        # Also get success info if available
        initial_success = group["initial_success"].iloc[0] if "initial_success" in group.columns else None
        
        shot_stats_with_cluster.append({
            "shot_id": shot_id,
            "cluster_frac_llr_2norm": cluster_frac,
            "best_llr": best_llr,
            "logical_gap": logical_gap,
            "min_llr": min_llr,
            "initial_success": initial_success,
        })
    
    df_cluster_gap = pd.DataFrame(shot_stats_with_cluster)
    
    print("=" * 60)
    print("CLUSTER FRACTION LLR 2-NORM vs LOGICAL GAP")
    print("=" * 60)
    
    print(f"\nCluster Fraction LLR 2-Norm:")
    print(f"  Mean:   {df_cluster_gap['cluster_frac_llr_2norm'].mean():.4f}")
    print(f"  Std:    {df_cluster_gap['cluster_frac_llr_2norm'].std():.4f}")
    print(f"  Range:  [{df_cluster_gap['cluster_frac_llr_2norm'].min():.4f}, {df_cluster_gap['cluster_frac_llr_2norm'].max():.4f}]")
    
    print(f"\nLogical Gap (2nd best LLR - best LLR):")
    print(f"  Mean:   {df_cluster_gap['logical_gap'].mean():.2f}")
    print(f"  Std:    {df_cluster_gap['logical_gap'].std():.2f}")
    print(f"  Range:  [{df_cluster_gap['logical_gap'].min():.2f}, {df_cluster_gap['logical_gap'].max():.2f}]")
    
    # Correlation analysis
    if len(df_cluster_gap) > 2:
        rho, p_val = stats.spearmanr(
            df_cluster_gap["cluster_frac_llr_2norm"], 
            df_cluster_gap["logical_gap"]
        )
        pearson_r, pearson_p = stats.pearsonr(
            df_cluster_gap["cluster_frac_llr_2norm"], 
            df_cluster_gap["logical_gap"]
        )
        
        print(f"\nCorrelation (Cluster Fraction vs Logical Gap):")
        print(f"  Spearman ρ: {rho:.4f} (p = {p_val:.2e})")
        print(f"  Pearson r:  {pearson_r:.4f} (p = {pearson_p:.2e})")
    
    # Scatter plot
    fig = go.Figure()
    
    # Color by initial_success if available
    if df_cluster_gap["initial_success"].notna().any():
        colors = df_cluster_gap["initial_success"].map({True: "green", False: "red"})
        color_labels = df_cluster_gap["initial_success"].map({True: "Success", False: "Fail"})
        
        for success_val, color, label in [(True, "green", "Initial Success"), (False, "red", "Initial Fail")]:
            mask = df_cluster_gap["initial_success"] == success_val
            if mask.any():
                fig.add_trace(go.Scatter(
                    x=df_cluster_gap.loc[mask, "cluster_frac_llr_2norm"],
                    y=df_cluster_gap.loc[mask, "logical_gap"],
                    mode="markers",
                    name=label,
                    marker=dict(color=color, size=12),
                    text=df_cluster_gap.loc[mask, "shot_id"],
                    hovertemplate="Shot %{text}<br>Cluster Frac: %{x:.4f}<br>Gap: %{y:.2f}<extra></extra>",
                ))
    else:
        fig.add_trace(go.Scatter(
            x=df_cluster_gap["cluster_frac_llr_2norm"],
            y=df_cluster_gap["logical_gap"],
            mode="markers",
            marker=dict(color="steelblue", size=12),
            text=df_cluster_gap["shot_id"],
            hovertemplate="Shot %{text}<br>Cluster Frac: %{x:.4f}<br>Gap: %{y:.2f}<extra></extra>",
        ))
    
    # Add correlation annotation
    if len(df_cluster_gap) > 2:
        fig.add_annotation(
            text=f"Spearman ρ = {rho:.3f}<br>p = {p_val:.2e}",
            xref="paper", yref="paper",
            x=0.02, y=0.98,
            showarrow=False,
            font=dict(size=12),
            bgcolor="white",
            bordercolor="gray",
            borderwidth=1,
            align="left",
        )
    
    fig.update_layout(
        title="Cluster Fraction LLR 2-Norm vs Logical Gap",
        xaxis_title="Cluster Fraction LLR 2-Norm (higher = more clustered)",
        yaxis_title="Logical Gap (2nd best - best LLR)",
        showlegend=True,
    )
    
    fig.show()
    
else:
    print("Note: cluster_frac_llr_2norm column not found in data.")

## 2.5 LLR vs Error Rank/Probability Visualization

Visualize how fixed-class LLR varies with error rank and probability across different shots.
- All classes (including no-error) are ranked by frequency from the distribution
- **Rank 0**: Most common class (typically no-error for low error rates)
- **Higher ranks**: Less common classes

In [ ]:
# ==============================================================================
# Figure 1: LLR vs Logical Error Rank (per shot)
# ==============================================================================

fig = go.Figure()

# Use a color scale for different shots
colors = px.colors.qualitative.Plotly
n_colors = len(colors)

for i, (shot_id, group) in enumerate(df_results.groupby("shot_id")):
    group_sorted = group.sort_values("error_rank")
    color = colors[i % n_colors]
    
    # Main line (all classes ranked by frequency)
    fig.add_trace(go.Scatter(
        x=group_sorted["error_rank"],
        y=group_sorted["fixed_llr"],
        mode="lines",
        name=f"Shot {shot_id}",
        line=dict(color=color, width=1.5),
        opacity=0.8,
        legendgroup=f"shot_{shot_id}",
    ))
    
    # Highlight the no-error case (error_index=0) with a marker
    no_error = group_sorted[group_sorted["error_index"] == 0]
    if len(no_error) > 0:
        fig.add_trace(go.Scatter(
            x=no_error["error_rank"],
            y=no_error["fixed_llr"],
            mode="markers",
            marker=dict(color=color, size=10, symbol="star"),
            showlegend=False,
            legendgroup=f"shot_{shot_id}",
            hovertemplate=f"Shot {shot_id}<br>No error (index=0)<br>Rank: %{{x}}<br>LLR: %{{y:.2f}}<extra></extra>",
        ))

fig.update_layout(
    title="Fixed-Class LLR vs Logical Class Rank (Each Line = One Shot)",
    xaxis_title="Class Rank (0 = most common class from distribution)",
    yaxis_title="Fixed-Class LLR",
    legend_title="Shot ID",
    hovermode="closest",
)

fig.show()

In [ ]:
# ==============================================================================
# Figure 2: LLR vs Logical Class Probability (per shot)
# ==============================================================================

fig = go.Figure()

for i, (shot_id, group) in enumerate(df_results.groupby("shot_id")):
    color = colors[i % n_colors]
    
    # Sort by probability (descending) for consistent line direction
    group_sorted = group.sort_values("error_prob", ascending=False)
    
    # Main line
    fig.add_trace(go.Scatter(
        x=group_sorted["error_prob"],
        y=group_sorted["fixed_llr"],
        mode="lines",
        name=f"Shot {shot_id}",
        line=dict(color=color, width=1.5),
        opacity=0.8,
        legendgroup=f"shot_{shot_id}",
    ))
    
    # Highlight the no-error case (error_index=0) with a marker
    no_error = group_sorted[group_sorted["error_index"] == 0]
    if len(no_error) > 0:
        fig.add_trace(go.Scatter(
            x=no_error["error_prob"],
            y=no_error["fixed_llr"],
            mode="markers",
            marker=dict(color=color, size=10, symbol="star"),
            showlegend=False,
            legendgroup=f"shot_{shot_id}",
            hovertemplate=f"Shot {shot_id}<br>No error (index=0)<br>Prob: %{{x:.4f}}<br>LLR: %{{y:.2f}}<extra></extra>",
        ))

fig.update_layout(
    title="Fixed-Class LLR vs Logical Class Probability (Each Line = One Shot)",
    xaxis_title="Class Probability (from distribution, ★ = no error)",
    yaxis_title="Fixed-Class LLR",
    legend_title="Shot ID",
    hovermode="closest",
    xaxis_type="log",  # Log scale for probability
)

fig.show()

## 3. Per-Shot Correlation Analysis

In [74]:
# ==============================================================================
# Compute Per-Shot Statistics
# ==============================================================================

per_shot_stats = []

for shot_id, group in df_results.groupby("shot_id"):
    # Sort by error rank to ensure consistent ordering
    group = group.sort_values("error_rank")
    
    # Per-shot Spearman correlation (rank vs LLR)
    rho, p_val = stats.spearmanr(group["error_rank"], group["fixed_llr"])
    
    # Find which rank achieved the minimum LLR
    min_idx = group["fixed_llr"].idxmin()
    min_llr_rank = group.loc[min_idx, "error_rank"]
    min_llr = group["fixed_llr"].min()
    
    # Rank-0 statistics (most likely error)
    rank0_llr = group[group["error_rank"] == 0]["fixed_llr"].values[0] if 0 in group["error_rank"].values else None
    rank0_is_best = (min_llr_rank == 0)
    
    # Regret: LLR penalty for using rank-0 instead of optimal
    regret = (rank0_llr - min_llr) if rank0_llr is not None else None
    
    # Baseline difficulty (best_llr from standard decoding)
    best_llr = group["best_llr"].iloc[0]
    
    # Mean LLR for random selection baseline
    mean_llr = group["fixed_llr"].mean()
    random_regret = mean_llr - min_llr
    
    per_shot_stats.append({
        "shot_id": shot_id,
        "spearman_rho": rho,
        "spearman_p": p_val,
        "min_llr_rank": min_llr_rank,
        "min_llr": min_llr,
        "rank0_llr": rank0_llr,
        "rank0_is_best": rank0_is_best,
        "regret": regret,
        "random_regret": random_regret,
        "best_llr": best_llr,
        "mean_fixed_llr": mean_llr,
    })

df_per_shot = pd.DataFrame(per_shot_stats)

print("=" * 60)
print("PER-SHOT STATISTICS COMPUTED")
print("=" * 60)
print(f"\nAnalyzed {len(df_per_shot)} shots")
print(f"Errors per shot: {n_errors_per_shot}")
print(f"\nDataFrame preview:")
df_per_shot

PER-SHOT STATISTICS COMPUTED

Analyzed 10 shots
Errors per shot: 4095

DataFrame preview:


,shot_id,spearman_rho,spearman_p,min_llr_rank,min_llr,rank0_llr,rank0_is_best,regret,random_regret,best_llr,mean_fixed_llr
0,0,0.023061,0.140089,1256,224.255747,923.608043,False,699.352296,312.590088,142.548704,536.845835
1,1,0.029696,0.057415,196,308.145241,532.779690,False,224.634449,231.979952,256.836494,540.125192
2,2,0.024999,0.109714,123,222.781207,523.894680,False,301.113473,250.367534,157.802434,473.148741
3,3,0.022473,0.150471,62,278.652613,606.987776,False,328.335163,188.643003,202.848009,467.295615
4,4,0.055940,0.000342,373,226.379198,465.721628,False,239.342430,197.667697,170.813508,424.046895
5,5,0.017848,0.253514,2952,251.475601,470.064663,False,218.589063,191.571786,191.333593,443.047387
6,6,0.034945,0.025336,91,271.067262,443.953958,False,172.886696,214.285338,205.420972,485.352600
7,7,0.031496,0.043862,1187,281.278055,569.395622,False,288.117567,198.544983,211.100405,479.823038
8,8,0.040395,0.009731,420,238.851112,387.458482,False,148.607370,177.136853,180.894186,415.987965
9,9,0.032782,0.035932,106,209.392217,712.511287,False,503.119070,308.756266,171.433871,518.148483


## 4. Per-Shot Spearman Correlation Statistics

In [75]:
# ==============================================================================
# Per-Shot Correlation Statistics
# ==============================================================================

print("=" * 60)
print("PER-SHOT SPEARMAN CORRELATION STATISTICS")
print("=" * 60)

# Filter out NaN correlations (can happen if all LLRs are identical in a shot)
valid_correlations = df_per_shot["spearman_rho"].dropna()
n_valid = len(valid_correlations)
n_total = len(df_per_shot)

print(f"\nValid correlations: {n_valid}/{n_total} shots")

# Basic statistics
mean_rho = valid_correlations.mean()
std_rho = valid_correlations.std()
median_rho = valid_correlations.median()

print(f"\nPer-shot Spearman correlation (rank vs LLR):")
print(f"  Mean:   {mean_rho:.4f}")
print(f"  Std:    {std_rho:.4f}")
print(f"  Median: {median_rho:.4f}")
print(f"  Min:    {valid_correlations.min():.4f}")
print(f"  Max:    {valid_correlations.max():.4f}")

# One-sample t-test: is mean correlation significantly different from 0?
if n_valid > 1:
    t_stat, t_pval = stats.ttest_1samp(valid_correlations, 0)
    print(f"\nOne-sample t-test (H0: mean = 0):")
    print(f"  t-statistic: {t_stat:.4f}")
    print(f"  p-value: {t_pval:.2e}")
    print(f"  Interpretation: {'Mean is significantly different from 0' if t_pval < 0.05 else 'Mean is NOT significantly different from 0'}")
else:
    t_pval = None
    print("\nInsufficient samples for t-test")

# Fraction of positive/negative correlations
n_positive = (valid_correlations > 0).sum()
n_negative = (valid_correlations < 0).sum()
n_zero = (valid_correlations == 0).sum()
print(f"\nCorrelation sign distribution:")
print(f"  Positive (rank ↑ → LLR ↑): {n_positive} ({100*n_positive/n_valid:.1f}%)")
print(f"  Negative (rank ↑ → LLR ↓): {n_negative} ({100*n_negative/n_valid:.1f}%)")
print(f"  Zero:                       {n_zero} ({100*n_zero/n_valid:.1f}%)")

PER-SHOT SPEARMAN CORRELATION STATISTICS

Valid correlations: 10/10 shots

Per-shot Spearman correlation (rank vs LLR):
  Mean:   0.0314
  Std:    0.0109
  Median: 0.0306
  Min:    0.0178
  Max:    0.0559

One-sample t-test (H0: mean = 0):
  t-statistic: 9.0776
  p-value: 7.96e-06
  Interpretation: Mean is significantly different from 0

Correlation sign distribution:
  Positive (rank ↑ → LLR ↑): 10 (100.0%)
  Negative (rank ↑ → LLR ↓): 0 (0.0%)
  Zero:                       0 (0.0%)


In [76]:
# ==============================================================================
# Histogram of Per-Shot Correlations
# ==============================================================================

fig = go.Figure()

# Histogram of per-shot Spearman correlations
fig.add_trace(go.Histogram(
    x=valid_correlations,
    nbinsx=max(10, n_valid // 2),
    name="Per-shot ρ",
    marker_color="steelblue",
    opacity=0.7,
))

# Add vertical line at mean
fig.add_vline(
    x=mean_rho,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean = {mean_rho:.3f}",
    annotation_position="top",
)

# Add vertical line at 0
fig.add_vline(
    x=0,
    line_dash="solid",
    line_color="black",
    line_width=1,
)

fig.update_layout(
    title="Distribution of Per-Shot Spearman Correlations (Rank vs LLR) - EXHAUSTIVE",
    xaxis_title="Spearman ρ (per shot)",
    yaxis_title="Count",
    showlegend=False,
)

# Add annotation with statistics
stats_text = f"Mean = {mean_rho:.3f}<br>Std = {std_rho:.3f}"
if t_pval is not None:
    stats_text += f"<br>t-test p = {t_pval:.2e}"
fig.add_annotation(
    text=stats_text,
    xref="paper", yref="paper",
    x=0.02, y=0.98,
    showarrow=False,
    font=dict(size=11),
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1,
    align="left",
)

fig.show()

## 5. Optimal Selection Rate & Regret Analysis

In [77]:
# ==============================================================================
# Optimal Selection Rate and Regret Analysis
# ==============================================================================

print("=" * 60)
print("OPTIMAL SELECTION RATE")
print("=" * 60)

random_baseline = 1.0 / n_errors_per_shot

# Rank-0 selection success rate
rank0_success_rate = df_per_shot["rank0_is_best"].mean()
print(f"\nIf we always select rank-0 (highest probability) error:")
print(f"  Success rate (rank-0 has min LLR): {100*rank0_success_rate:.2f}%")
print(f"  Random baseline:                   {100*random_baseline:.4f}%")
print(f"  Improvement over random:           {100*(rank0_success_rate - random_baseline):.2f}pp")
print(f"  Improvement factor:                {rank0_success_rate / random_baseline:.1f}x")

# Binomial test: is success rate better than random?
if n_shots > 1:
    binom_result = stats.binomtest(
        k=int(df_per_shot["rank0_is_best"].sum()),
        n=len(df_per_shot),
        p=random_baseline,
        alternative="greater",
    )
    print(f"\nBinomial test (H0: success rate = random):")
    print(f"  p-value: {binom_result.pvalue:.2e}")
    print(f"  95% CI: [{binom_result.proportion_ci(confidence_level=0.95).low:.4f}, {binom_result.proportion_ci(confidence_level=0.95).high:.4f}]")

print("\n" + "=" * 60)
print("REGRET ANALYSIS")
print("=" * 60)

valid_regrets = df_per_shot["regret"].dropna()
mean_regret = valid_regrets.mean()
mean_random_regret = df_per_shot["random_regret"].mean()

print(f"\nRegret = LLR(selected) - LLR(optimal)")
print(f"\nRank-0 selection regret:")
print(f"  Mean:   {mean_regret:.2f}")
print(f"  Median: {valid_regrets.median():.2f}")
print(f"  Std:    {valid_regrets.std():.2f}")

print(f"\nRandom selection regret (expected):")
print(f"  Mean:   {mean_random_regret:.2f}")
print(f"  Median: {df_per_shot['random_regret'].median():.2f}")
print(f"  Std:    {df_per_shot['random_regret'].std():.2f}")

print(f"\nRegret reduction (random → rank-0): {mean_random_regret - mean_regret:.2f}")

OPTIMAL SELECTION RATE

If we always select rank-0 (highest probability) error:
  Success rate (rank-0 has min LLR): 0.00%
  Random baseline:                   0.0244%
  Improvement over random:           -0.02pp
  Improvement factor:                0.0x

Binomial test (H0: success rate = random):
  p-value: 1.00e+00
  95% CI: [0.0000, 1.0000]

REGRET ANALYSIS

Regret = LLR(selected) - LLR(optimal)

Rank-0 selection regret:
  Mean:   312.41
  Median: 263.73
  Std:    168.39

Random selection regret (expected):
  Mean:   227.15
  Median: 206.42
  Std:    49.02

Regret reduction (random → rank-0): -85.26


## 6. Complete Top-k Success Rate Curve

In [78]:
# ==============================================================================
# Complete Top-k Success Rate (Exhaustive Analysis)
# ==============================================================================

print("=" * 60)
print("COMPLETE TOP-K SUCCESS RATE")
print("=" * 60)
print("\nIf we pick among top-k errors by distribution probability,")
print("how often does the true minimum LLR fall within that set?")
print()

unique_ranks = sorted(df_results["error_rank"].unique())
n_total_ranks = len(unique_ranks)

topk_results = []
for k in range(1, min(n_total_ranks + 1, 101)):  # Cap at 100 for display
    top_k_ranks = set(unique_ranks[:k])
    success_count = (df_per_shot["min_llr_rank"].isin(top_k_ranks)).sum()
    success_rate = success_count / len(df_per_shot)
    random_rate = k / n_total_ranks
    topk_results.append({
        "k": k,
        "success_rate": success_rate,
        "random_rate": random_rate,
        "improvement": success_rate - random_rate,
    })

df_topk = pd.DataFrame(topk_results)

# Show summary (first and last 10)
print("First 10 k values:")
print(df_topk.head(10).to_string(index=False, formatters={
    "success_rate": "{:.2%}".format,
    "random_rate": "{:.2%}".format,
    "improvement": "{:+.2%}".format,
}))

if len(df_topk) > 20:
    print("\n...\n")
    print("Last 10 k values:")
    print(df_topk.tail(10).to_string(index=False, formatters={
        "success_rate": "{:.2%}".format,
        "random_rate": "{:.2%}".format,
        "improvement": "{:+.2%}".format,
    }))

COMPLETE TOP-K SUCCESS RATE

If we pick among top-k errors by distribution probability,
how often does the true minimum LLR fall within that set?

First 10 k values:
 k success_rate random_rate improvement
 1        0.00%       0.02%      -0.02%
 2        0.00%       0.05%      -0.05%
 3        0.00%       0.07%      -0.07%
 4        0.00%       0.10%      -0.10%
 5        0.00%       0.12%      -0.12%
 6        0.00%       0.15%      -0.15%
 7        0.00%       0.17%      -0.17%
 8        0.00%       0.20%      -0.20%
 9        0.00%       0.22%      -0.22%
10        0.00%       0.24%      -0.24%

...

Last 10 k values:
  k success_rate random_rate improvement
 91       10.00%       2.22%      +7.78%
 92       20.00%       2.25%     +17.75%
 93       20.00%       2.27%     +17.73%
 94       20.00%       2.30%     +17.70%
 95       20.00%       2.32%     +17.68%
 96       20.00%       2.34%     +17.66%
 97       20.00%       2.37%     +17.63%
 98       20.00%       2.39%     +17.61%
 

In [86]:
# ==============================================================================
# Visualization: Complete Top-k Success Rate
# ==============================================================================

fig = go.Figure()

# Actual success rate
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["success_rate"],
    mode="lines+markers" if len(df_topk) <= 50 else "lines",
    name="Actual (distribution-guided)",
    line=dict(color="steelblue", width=2),
    marker=dict(size=6),
))

# Random baseline
fig.add_trace(go.Scatter(
    x=df_topk["k"],
    y=df_topk["random_rate"],
    mode="lines",
    name="Random baseline",
    line=dict(color="red", width=2, dash="dash"),
))

fig.update_layout(
    title="Complete Top-k Success Rate: Does Distribution Rank Predict Winner? (EXHAUSTIVE)",
    xaxis_title="k (number of top ranks considered)",
    yaxis_title="Success Rate (min LLR in top-k)",
    yaxis_tickformat=".0%",
    legend=dict(x=0.6, y=0.1),
    # xaxis_type="log" if n_total_ranks > 100 else "linear",
)

fig.show()

## 7. Distribution of Winning Rank

In [80]:
# ==============================================================================
# Distribution of Winning Rank
# ==============================================================================

print("=" * 60)
print("DISTRIBUTION OF WINNING RANK")
print("=" * 60)
print("\nWhich rank achieves the minimum LLR most often?")
print()

win_counts = df_per_shot["min_llr_rank"].value_counts().sort_index()
expected_pct = 100.0 / n_total_ranks

# Show top 20 winning ranks
top_winners = win_counts.head(20)

winner_df = pd.DataFrame({
    "rank": top_winners.index,
    "wins": top_winners.values,
    "win_pct": top_winners.values / len(df_per_shot) * 100,
    "expected_pct": expected_pct,
})
winner_df["vs_expected"] = winner_df["win_pct"] - expected_pct

print(f"Top 20 winning ranks (expected: {expected_pct:.4f}% each):")
print(winner_df.to_string(index=False, formatters={
    "win_pct": "{:.2f}%".format,
    "expected_pct": "{:.4f}%".format,
    "vs_expected": "{:+.2f}pp".format,
}))

print(f"\nTotal unique winning ranks: {len(win_counts)} / {n_total_ranks}")

DISTRIBUTION OF WINNING RANK

Which rank achieves the minimum LLR most often?

Top 20 winning ranks (expected: 0.0244% each):
 rank  wins win_pct expected_pct vs_expected
   62     1  10.00%      0.0244%     +9.98pp
   91     1  10.00%      0.0244%     +9.98pp
  106     1  10.00%      0.0244%     +9.98pp
  123     1  10.00%      0.0244%     +9.98pp
  196     1  10.00%      0.0244%     +9.98pp
  373     1  10.00%      0.0244%     +9.98pp
  420     1  10.00%      0.0244%     +9.98pp
 1187     1  10.00%      0.0244%     +9.98pp
 1256     1  10.00%      0.0244%     +9.98pp
 2952     1  10.00%      0.0244%     +9.98pp

Total unique winning ranks: 10 / 4095


In [81]:
# ==============================================================================
# Visualization: Winning Rank Distribution
# ==============================================================================

fig = go.Figure()

# Bar chart of win percentages (top 50 or all if fewer)
n_display = min(50, len(win_counts))
top_winners = win_counts.head(n_display)

fig.add_trace(go.Bar(
    x=[str(r) for r in top_winners.index],
    y=top_winners.values / len(df_per_shot) * 100,
    name="Win %",
    marker_color="steelblue",
))

# Random baseline
fig.add_hline(
    y=expected_pct,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Random: {expected_pct:.4f}%",
    annotation_position="top right",
)

fig.update_layout(
    title=f"Which Rank Wins Most Often? (Top {n_display} shown)",
    xaxis_title="Error Rank",
    yaxis_title="Win Percentage (%)",
    showlegend=False,
)

fig.show()

## 8. Conditional Analysis by Shot Difficulty

In [82]:
# ==============================================================================
# Conditional Analysis by Shot Difficulty
# ==============================================================================

print("=" * 60)
print("CONDITIONAL ANALYSIS BY SHOT DIFFICULTY")
print("=" * 60)
print("\nDoes correlation vary with shot difficulty (baseline LLR)?")

if n_shots >= 4:
    # Stratify by best_llr quartiles
    df_per_shot["difficulty_quartile"] = pd.qcut(
        df_per_shot["best_llr"], 
        q=min(4, n_shots),
        labels=["Q1 (easy)", "Q2", "Q3", "Q4 (hard)"][:min(4, n_shots)]
    )

    # Compute statistics per quartile
    quartile_stats = df_per_shot.groupby("difficulty_quartile", observed=True).agg({
        "spearman_rho": ["mean", "std", "count"],
        "rank0_is_best": "mean",
        "regret": "mean",
        "best_llr": ["min", "max"],
    }).round(4)

    quartile_stats.columns = [
        "rho_mean", "rho_std", "n_shots",
        "rank0_success", "mean_regret",
        "llr_min", "llr_max"
    ]

    print("\nPer-quartile statistics:")
    print(quartile_stats.to_string())
else:
    print(f"\nInsufficient shots ({n_shots}) for quartile analysis. Need at least 4 shots.")

CONDITIONAL ANALYSIS BY SHOT DIFFICULTY

Does correlation vary with shot difficulty (baseline LLR)?

Per-quartile statistics:
                     rho_mean  rho_std  n_shots  rank0_success  mean_regret    llr_min    llr_max
difficulty_quartile                                                                              
Q1 (easy)            0.034700 0.018400        3       0.000000   413.269400 142.548700 170.813500
Q2                   0.036600 0.005400        2       0.000000   325.863200 171.433900 180.894200
Q3                   0.020200 0.003300        2       0.000000   273.462100 191.333600 202.848000
Q4 (hard)            0.032000 0.002700        3       0.000000   228.546200 205.421000 256.836500


## 9. Summary & Conclusions

In [83]:
print("=" * 60)
print("SUMMARY (EXHAUSTIVE ANALYSIS)")
print("=" * 60)

print(f"\n--- Code & Analysis Parameters ---")
print(f"BB code: [[{metadata.get('n_qubits', N_QUBITS)}, 12, {metadata.get('distance', '?')}]]")
print(f"Physical error rate: {metadata.get('p', P)}")
print(f"Total logical classes: {metadata.get('total_logical_classes', total_logical_classes)}")
print(f"Shots analyzed: {n_shots}")
print(f"Errors per shot: {n_errors_per_shot}")

print(f"\n--- Per-Shot Correlation ---")
print(f"Mean Spearman ρ: {mean_rho:.4f} ± {std_rho:.4f}")
if t_pval is not None:
    print(f"Significantly different from 0: {'Yes' if t_pval < 0.05 else 'No'} (p = {t_pval:.2e})")

print(f"\n--- Rank-0 Selection (most likely error) ---")
print(f"Success rate: {100*rank0_success_rate:.2f}%")
print(f"Random baseline: {100*random_baseline:.4f}%")
print(f"Improvement factor: {rank0_success_rate / random_baseline:.1f}x")

print(f"\n--- Regret ---")
print(f"Mean regret (rank-0): {mean_regret:.2f}")
print(f"Mean regret (random): {mean_random_regret:.2f}")
print(f"Regret reduction: {mean_random_regret - mean_regret:.2f}")

print("\n" + "=" * 60)
print("INTERPRETATION")
print("=" * 60)

# Interpret based on results
if abs(mean_rho) < 0.1:
    print("\n• Per-shot correlation is NEGLIGIBLE (|ρ| < 0.1)")
    print("  → Distribution rank does NOT strongly predict LLR ordering within shots")
elif mean_rho > 0.1:
    print(f"\n• Per-shot correlation is POSITIVE (ρ = {mean_rho:.3f})")
    print("  → Higher rank tends to have higher LLR (distribution is useful)")
else:
    print(f"\n• Per-shot correlation is NEGATIVE (ρ = {mean_rho:.3f})")
    print("  → Higher rank tends to have LOWER LLR (unexpected)")

if rank0_success_rate > random_baseline * 2:
    print(f"\n• Rank-0 selection has STRONG advantage ({rank0_success_rate/random_baseline:.1f}x random)")
elif rank0_success_rate > random_baseline * 1.2:
    print(f"\n• Rank-0 selection has MODERATE advantage ({rank0_success_rate/random_baseline:.1f}x random)")
else:
    print(f"\n• Rank-0 selection has MINIMAL/NO advantage ({rank0_success_rate/random_baseline:.1f}x random)")

SUMMARY (EXHAUSTIVE ANALYSIS)

--- Code & Analysis Parameters ---
BB code: [[144, 12, 12]]
Physical error rate: 0.003
Total logical classes: 4096
Shots analyzed: 10
Errors per shot: 4095

--- Per-Shot Correlation ---
Mean Spearman ρ: 0.0314 ± 0.0109
Significantly different from 0: Yes (p = 7.96e-06)

--- Rank-0 Selection (most likely error) ---
Success rate: 0.00%
Random baseline: 0.0244%
Improvement factor: 0.0x

--- Regret ---
Mean regret (rank-0): 312.41
Mean regret (random): 227.15
Regret reduction: -85.26

INTERPRETATION

• Per-shot correlation is NEGLIGIBLE (|ρ| < 0.1)
  → Distribution rank does NOT strongly predict LLR ordering within shots

• Rank-0 selection has MINIMAL/NO advantage (0.0x random)
